In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
print("Done")

Done


In [11]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Fix hidden missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop useless column
df.drop('customerID', axis=1, inplace=True)

print("Loaded:", df.shape)
print("Missing values:", df.isnull().sum().sum())

Loaded: (7043, 20)
Missing values: 0


In [12]:
# Binning tenure into groups
df['tenure_bin'] = pd.cut(df['tenure'], bins=[0,12,36,72], labels=['new','mid','loyal'])

# Average monthly spend
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)

# Convert target to 0/1
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# One-hot encode ALL text/categorical columns
df = pd.get_dummies(df, drop_first=True)

# Make sure all columns are numeric (safety check)
df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

X = df.drop('Churn', axis=1)
y = df['Churn']

print("Features:", X.shape[1])
print("Churn balance - No:", (y==0).sum(), "Yes:", (y==1).sum())

Features: 33
Churn balance - No: 5174 Yes: 1869


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify keeps class balance in split
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)   # IMPORTANT: only transform test, never fit on it

print("Train size:", X_train_sc.shape)
print("Test size:", X_test_sc.shape)

Train size: (5634, 33)
Test size: (1409, 33)


In [16]:
# class_weight='balanced' fixes the imbalance problem for LR, RF, SVM
# scale_pos_weight fixes it for XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'XGBoost':             XGBClassifier(scale_pos_weight=neg/pos, eval_metric='logloss', random_state=42),
    'SVM (RBF kernel)':    SVC(kernel='rbf', class_weight='balanced', probability=True)
}

results = {}
print("=== Model Comparison (5-Fold Cross-Validation) ===\n")

for name, model in models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='f1')
    results[name] = scores.mean()
    print(f"{name:<25} F1 = {scores.mean():.3f}  (+/- {scores.std():.3f})")

best_name = max(results, key=results.get)
print(f"\nBest model: {best_name}  (F1 = {results[best_name]:.3f})")

=== Model Comparison (5-Fold Cross-Validation) ===

Logistic Regression       F1 = 0.634  (+/- 0.025)
Random Forest             F1 = 0.553  (+/- 0.025)
XGBoost                   F1 = 0.588  (+/- 0.027)
SVM (RBF kernel)          F1 = 0.622  (+/- 0.022)

Best model: Logistic Regression  (F1 = 0.634)


In [17]:
print("=== Hyperparameter Tuning (GridSearchCV) ===\n")

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    param_grid, cv=5, scoring='f1'
)
grid_search.fit(X_train_sc, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1 Score:", round(grid_search.best_score_, 3))

In [18]:
print("=== Final Evaluation on Test Set ===\n")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_sc)
y_prob = best_model.predict_proba(X_test_sc)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.3f}")

In [19]:
print("=== Top 5 Feature Importances (Coefficients) ===\n")

importances = pd.Series(best_model.coef_[0], index=X.columns).sort_values(ascending=False)
print(importances.head(5))
print("\n... and top negative indicators (most likely to stay):")
print(importances.tail(5))

In [20]:
import joblib
import os

# Ensure directory exists before saving
os.makedirs('models', exist_ok=True)

model_path = 'models/churn_lr_model.pkl'
joblib.dump(best_model, model_path)
print(f"\nModel saved successfully to {model_path}")